In [1]:
from bigmodule import M

# <aistudiograph>

# @param(id="m1", name="initialize")
# 交易引擎：初始化函数，只执行一次
def m1_initialize_bigquant_run(context):
    from bigtrader.finance.commission import PerOrder

    # 系统已经设置了默认的交易手续费和滑点，要修改手续费可使用如下函数
    context.set_commission(PerOrder(buy_cost=0.0003, sell_cost=0.0013, min_cost=5))

# @param(id="m1", name="before_trading_start")
# 交易引擎：每个单位时间开盘前调用一次。
def m1_before_trading_start_bigquant_run(context, data):
    # 盘前处理，订阅行情等
    pass

# @param(id="m1", name="handle_tick")
# 交易引擎：tick数据处理函数，每个tick执行一次
def m1_handle_tick_bigquant_run(context, tick):
    pass

# @param(id="m1", name="handle_data")
def m1_handle_data_bigquant_run(context, data):
    import pandas as pd

    # 下一个交易日不是调仓日，则不生成信号
    if not context.rebalance_period.is_signal_date(data.current_dt.date()):
        return

    # 从传入的数据 context.data 中读取今天的信号数据
    today_df = context.data[context.data["date"] == data.current_dt.strftime("%Y-%m-%d")]
    print(f"handel_data({data.current_dt}), today_df=\n", today_df)
    target_instruments = set(today_df["instrument"])

    # 获取当前已持有股票
    holding_instruments = set(context.get_account_positions().keys())
    print(f"handel_data({data.current_dt}), target_instruments={target_instruments}, holding_instruments={holding_instruments}")

    # 卖出不在目标持有列表中的股票
    for instrument in holding_instruments - target_instruments:
        context.order_target_percent(instrument, 0)
        
    # 买入目标持有列表中的股票
    for i, x in today_df.iterrows():
        # 处理 null 或者 decimal.Decimal 类型等
        position = 0.0 if pd.isnull(x.position) else float(x.position)
        rv = context.order_target_percent(x.instrument, position)
        print(f"handel_data({data.current_dt}), order_target_percent({x.instrument},{position}), rv={rv}")

# @param(id="m1", name="handle_trade")
# 交易引擎：成交回报处理函数，每个成交发生时执行一次
def m1_handle_trade_bigquant_run(context, trade):
    pass

# @param(id="m1", name="handle_order")
# 交易引擎：委托回报处理函数，每个委托变化时执行一次
def m1_handle_order_bigquant_run(context, order):
    pass

# @param(id="m1", name="after_trading")
# 交易引擎：盘后处理函数，每日盘后执行一次
def m1_after_trading_bigquant_run(context, data):
    pass

# @module(position="-642,-1705", comment="""""", comment_collapsed=True)
m6 = M.input_features_dai.v30(
    mode="""表达式""",
    expr="""position""",
    expr_filters="""-- DAI SQL 算子/函数: https://bigquant.com/wiki/doc/dai-PLSbc1SbZX#h-%E5%87%BD%E6%95%B0
-- 数据&字段: 数据文档 https://bigquant.com/data/home
-- 表达式模式的过滤都是放在 QUALIFY 里, 即数据查询、计算, 最后才到过滤条件

-- c_pct_rank(-return_90) <= 0.3
-- c_pct_rank(return_30) <= 0.3
-- cn_stock_bar1d.turn > 0.02
""",
    expr_tables="""data_xs1sxs123gh""",
    extra_fields="""date, instrument""",
    order_by="""date, instrument""",
    expr_drop_na=True,
    sql="""-- 使用DAI SQL获取数据, 构建因子等, 如下是一个例子作为参考
-- DAI SQL 语法: https://bigquant.com/wiki/doc/dai-PLSbc1SbZX#h-sql%E5%85%A5%E9%97%A8%E6%95%99%E7%A8%8B
-- 使用数据输入1/2/3里的字段: e.g. input_1.close, input_1.* EXCLUDE(date, instrument)

SELECT
    -- 在这里输入因子表达式
    -- DAI SQL 算子/函数: https://bigquant.com/wiki/doc/dai-PLSbc1SbZX#h-%E5%87%BD%E6%95%B0
    -- 数据&字段: 数据文档 https://bigquant.com/data/home

    m_lag(close, 90) / close AS return_90,
    m_lag(close, 30) / close AS return_30,
    -- 下划线开始命名的列是中间变量, 不会在最终结果输出 (e.g. _rank_return_90)
    c_pct_rank(-return_90) AS _rank_return_90,
    c_pct_rank(return_30) AS _rank_return_30,

    c_rank(volume) AS rank_volume,
    close / m_lag(close, 1) as return_0,

    -- 日期和股票代码
    date, instrument
FROM
    -- 预计算因子 cn_stock_bar1d https://bigquant.com/data/datasources/cn_stock_bar1d
    cn_stock_prefactors
    -- SQL 模式不会自动join输入数据源, 可以根据需要自由灵活的使用
    -- JOIN input_1 USING(date, instrument)
WHERE
    -- WHERE 过滤, 在窗口等计算算子之前执行
    -- 剔除ST股票
    st_status = 0
QUALIFY
    -- QUALIFY 过滤, 在窗口等计算算子之后执行, 比如 m_lag(close, 3) AS close_3, 对于 close_3 的过滤需要放到这里
    -- 去掉有空值的行
    COLUMNS(*) IS NOT NULL
    -- _rank_return_90 是窗口函数结果，需要放在 QUALIFY 里
    AND _rank_return_90 > 0.1
    AND _rank_return_30 < 0.1
-- 按日期和股票代码排序, 从小到大
ORDER BY date, instrument
""",
    extract_data=False,
    m_name="""m6"""
)

# @module(position="-653,-1606", comment="""抽取预测数据""")
m3 = M.extract_data_dai.v17(
    sql=m6.data,
    start_date="""2022-10-25""",
    start_date_bound_to_trading_date=False,
    end_date="""2024-12-10""",
    end_date_bound_to_trading_date=True,
    before_start_days=0,
    debug=False,
    m_name="""m3"""
)

# @module(position="-666,-1487", comment="""交易，日线，设置初始化函数和K线处理函数，以及初始资金、基准等""", comment_collapsed=True)
m1 = M.bigtrader.v34(
    data=m3.data,
    start_date="""""",
    end_date="""""",
    initialize=m1_initialize_bigquant_run,
    before_trading_start=m1_before_trading_start_bigquant_run,
    handle_tick=m1_handle_tick_bigquant_run,
    handle_data=m1_handle_data_bigquant_run,
    handle_trade=m1_handle_trade_bigquant_run,
    handle_order=m1_handle_order_bigquant_run,
    after_trading=m1_after_trading_bigquant_run,
    capital_base=150000,
    frequency="""daily""",
    product_type="""基金""",
    rebalance_period_type="""周度交易日""",
    rebalance_period_days="""1""",
    rebalance_period_roll_forward=True,
    backtest_engine_mode="""标准模式""",
    before_start_days=0,
    volume_limit=1,
    order_price_field_buy="""open""",
    order_price_field_sell="""open""",
    benchmark="""沪深300指数""",
    plot_charts=True,
    debug=True,
    backtest_only=False,
    m_name="""m1"""
)
# </aistudiograph>

[2026-06-25 15:37:50] [info     ] input_features_dai.v30 开始运行 ..
[2026-06-25 15:37:51] [info     ] input_features_dai.v30 命中缓存
[2026-06-25 15:37:51] [info     ] input_features_dai.v30 运行完成 [1.069s].
[2026-06-25 15:37:51] [info     ] extract_data_dai.v17 开始运行 ..
[2026-06-25 15:37:52] [warning  ] start_date='2022-10-25', end_date='2024-12-10', query_start_date='2022-10-25 00:00:00' (支持加速 [url="command:switch-quota"]升级资源[/url]) ..
[2026-06-25 15:37:52] [info     ] data extracted: (4144, 3)
[2026-06-25 15:37:52] [info     ] extract_data_dai.v17 运行完成 [0.189s].
[2026-06-25 15:37:53] [info     ] bigtrader.v34 开始运行 ..
[2026-06-25 15:37:53] [info     ] got metadata extra from input datasource
[2026-06-25 15:37:53] [info     ] read input 'data' ..
[2026-06-25 15:37:53] [info     ] 2022-10-25, 2024-12-10, , fund, instruments=8
[2026-06-25 15:37:54] [info     ] bigtrader module V2.2.0
[2026-06-25 15:37:54] [info     ] bigtrader engine v0.1.0.post9+g6d7300d 2026-02-10
[2026-06-25 15:37:54] [info   

[2026-06-25 15:38:00] [info     ] bigtrader.v34 运行完成 [7.658s].
